In [ ]:
from typing import Iterable
from itertools import chain as iterchain, combinations as itercomb
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Loc, Cell, Board
from topology import Zone, Node
from utils import count_digits, count_finals, draftborhood, draftboard, validate
from solving import solver, Resolution, Resolving, Resolver, solve_logging, solve_silent
from searching import search_breadth

In [ ]:
def filt_finals(c: Cell):
    return c.is_final


def filt_drafts(c: Cell):
    return c.is_draft


def filt_having(d: int):
    def filt(c: Cell):
        return d in c.digits

    return filt


def flat_cells(cc: Iterable[Cell]):
    return iterflat(c.digits for c in cc)

## Singles

A single is a draft about some digit with no concurrent drafts about the same digit in a unit.

A final is a single inside a cell == solution for the cell

Rules:

- (open singles) remove all draft conflicting with finals in all units
- (hidden singles) remove all other drafts of the same digit from other units


In [ ]:
def open_singles(board: Board) -> Resolving:
    """Open singles (finals): removing drafts conflicting with finals"""

    for fincell in filter(filt_finals, iter(board)):
        findig: int = fincell.final  # type: ignore
        for zone in Zone.around(Zone.of(fincell)):
            spoilers = tuple(filter(lambda c: c != fincell and findig in c, draftborhood(board, zone)))
            if len(spoilers):
                yield Resolution(
                    castaways={Node.at(c, findig) for c in spoilers},
                    highlights={"anchors": {Node.at(fincell, findig)}, "zone": zone},
                )

In [ ]:
def hidden_singles(board: Board) -> Resolving:
    """Hidden finals: removing all other drafts from the cell of a single"""

    for zone in Zone.All():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 1:
                (lonesome,) = filter(filt_having(dig), drafts)
                final = Node.at(lonesome, dig)
                yield Resolution(
                    finals={final},
                    highlights={
                        "zone": zone,
                        "anchors": {final},
                        "empties": {Node(zone, dig)},
                    },
                )

## Groups

Groups are set of drafts about same digit in a unit.
The draft in a group relates as OR

$G = (G_1 \vee G_2 \vee …)$

All the logic of grouped drafts is the same as for single-cell drafts.

The logic actually works only for groups at intersection of units.


In [ ]:
from topology import Group

In [ ]:
def scan_groups(board: Board, dig: int):
    for box in Zone.Allbox():
        for side in Zone.across(box):
            zone: Zone = box & side  # type: ignore impossible None
            drafts = frozenset(filter(lambda c: dig in c, draftborhood(board, zone)))
            if len(drafts):
                yield Group((box, side), dig, drafts)

### locked

A group which is alone in a unit is always true.

Rule is the same as for open singles:

- remove all conflicting drafts (in all units where the group is fully visible)


In [ ]:
def resolve_locked(board: Board, group: Group):
    # print(tuple(map(str, group.zones)), tuple(map(str, group.cells)))

    sidebours = {
        z: tuple(
            filter(
                lambda c: group.dig in c and c not in group.cells,
                draftborhood(board, z),
            )
        )
        for z in group.zones
    }

    counts = Counter({z: len(sidebours[z]) for z in group.zones})

    # for z in group.zones:
    #     print(z, list(map(str, sidebours[z])))

    # the last one is where it's possibly = 0
    ((z1, cnt1), (z0, cnt0)) = counts.most_common()

    if cnt0 == 0 and cnt1 > 0:
        yield Resolution(
            castaways={Node.at(c, group.dig) for c in sidebours[z1]},
            highlights={"empties": {Node(z0, group.dig)}, "anchors": {group.node()}},
        )


def locked_groups(board: Board) -> Resolving:
    counts = count_digits(draftboard(board))

    for dig, cnt in reversed(counts.most_common()):
        # print(dig, cnt, "...")
        for grp in scan_groups(board, dig):
            # print("...", *map(str, grp.cells))
            if len(grp.cells) == 1:
                continue
            yield from resolve_locked(board, grp)

In [ ]:
for _ in locked_groups(puzzle):
    pprint(_)

## Multiples

Combos of N digits


#### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

Rule: remove the digits of the combo from all other cells


In [ ]:
def open_mults(board: Board, mult: int) -> Resolving:
    """Clean out spoiled neighbours of open multiples in each zone"""
    for zone in Zone.All():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = set(cmb)
            # all cells containing only the combo
            habitat = set(filter(lambda c: c.digits <= combo, drafts))
            # all other neighbors containing some combo digits
            spoilers = tuple(filter(lambda c: c.digits & combo, drafts - habitat))

            if len(habitat) == mult and len(spoilers):
                yield Resolution(
                    castaways={Node.at(c, d) for c in spoilers for d in c.digits & combo},
                    highlights={
                        "zone": zone,
                        "anchors": {Node.at(c, d) for c in habitat for d in c.digits & combo},
                    },
                )


def open_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return open_mults(board, mult)

    resolver.__name__ = f"open_mults[{mult}]"
    return resolver

### hidden

Some n-combo contained in only n cells (within a unit) // along other drafts

Rule: remove all other drafts from the cells => it becomes open


In [ ]:
def hidden_mults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Zone.All():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = set(cmb)
            if len(inhabitants & combo) != mult:
                continue
            # all cells containing some combo digits (+ some spoilers)
            habitat = tuple(filter(lambda c: c.digits & combo, drafts))
            # inhabited cells with other digits
            spoiled = tuple(filter(lambda c: c.digits - combo, habitat))
            if len(habitat) == mult and len(spoiled):
                yield Resolution(
                    castaways={Node.at(c, d) for c in spoiled for d in c.digits - combo},
                    highlights={
                        "zone": zone,
                        "anchors": set(Node.at(c, d) for c in habitat for d in c.digits & combo),
                        "empties": {Node(zone, d) for d in combo},
                    },
                )


def hidden_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return hidden_mults(board, mult)

    resolver.__name__ = f"hidden_mults[{mult}]"
    return resolver

## Links

Links represent XOR or NAND relations between drafts.

- XOR $\veebar$ corresponds to "each digit appears only once in a locality"
- NAND $\barwedge$ corrsponds to "each locality contains only different digits"

(or vise versa, I dunno)


In [ ]:
from topology import Link, HLink, SLink, visibility, allvisible

#### strong/hard links

Represent XOR relation $\veebar$

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts (of different digits) in a cell


In [ ]:
def search_hard_segms(board: Board) -> Iterable[HLink]:
    """Search for all intra-cellular links (open pairs)"""
    for cell in draftboard(board):
        if len(cell) == 2:
            d1, d2 = cell.digits
            yield HLink((Node.at(cell, d1), Node.at(cell, d2)))


def search_hard_cells(board: Board) -> Iterable[HLink]:
    """Search for all inter-cellular links (same-digit)"""
    for zone in Zone.All():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(n1, dig), Node.at(n2, dig)))

#### weak/soft links

Represent NAND relation $\barwedge$

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts (of different digits) in a cell

Note: The criteria are totally independent of board content (calculating from locations only)

Visibility = soft-linkability


In [ ]:
def check_soft(n1: Node, n2: Node):
    assert n1 != n2
    if n1.dig != n2.dig:
        # different digits within a cell
        return n1.is_cellular and n2.is_cellular and n1.zone == n2.zone
    else:
        # same digits in some shared zone
        return len(set(visibility(n1.zone, n2.zone))) > 0

## Chains

Alterating link chains constituted of `~ hard ~ soft ~` and `~ soft ~ hard ~`

Lemma1: $(X \barwedge A) \cdot (A \veebar B) \cdot (B \barwedge X) \Rightarrow \neg X$

Meaning: all draft visible (soft-linkable) from some XORed points, are all invalid

Lemma2: $(X \veebar A) \cdot (A \barwedge B) \cdot (B \veebar Y) \Rightarrow (X \veebar Y)$

Meaning: a ALC (of any length) with hard edges behaves as if its edges are hard-linked


In [ ]:
from topology import Chain

In [ ]:
def search_soft(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    """Scan for all nodes nand-able with both e1 and e2"""
    interest: set[int] = {n1.dig, n2.dig}  # max=2

    for vizone in allvisible(n1.zone, n2.zone):  # max=2
        for cell in filter(lambda c: c.digits & interest, draftborhood(board, vizone)):  # max=18
            for dig in interest:  # max=36
                node = Node.at(cell, dig)
                if node == n1 or node == n2:
                    continue
                if check_soft(node, n1) and check_soft(node, n2):
                    yield node

#### loop ALC

ALC with connected edges: `X ~ hard ~ ... ~ soft ~ X`

Rule: invalidate all draft visible from each soft-link in the chain


In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors: set[Node] = chain.anchors()  # to exclude linking back to chain

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(search_soft(board, t1, t2)) - anchors
        if len(spoilers):
            yield Resolution(
                castaways=spoilers,
                highlights={"anchors": {t1, t2}, "chain": chain},
            )

#### open ALC

ALC with hard links at its edges: `X ~ hard ~ ... ~ hard ~ Y`

Rule: invalidate all drafts visible from both edges of such chain


In [ ]:
def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    anchors = chain.anchors()
    e1, e2 = chain.edges

    spoilers = set(search_soft(board, e1, e2)) - anchors
    if len(spoilers):
        yield Resolution(
            castaways=spoilers,
            highlights={"anchors": {e1, e2}, "chain": chain},
        )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [ ]:
def expand_alc(current: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and check_soft(e1, e2):
            yield Chain.extend(chain, SLink((e2, e1)))

    def stretch(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if check_soft(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if check_soft(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())
        if check_soft(x2, e1):
            yield Chain.extendhead(chain, link, SLink((x2, e1)))
        if check_soft(x1, e1):
            yield Chain.extendhead(chain, link.reversed(), SLink((x1, e1)))

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        for extended in stretch(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [ ]:
MAX_LENGTH = 8


def chains(current: Board) -> Resolving:
    links = set(search_hard_segms(current)) | set(search_hard_cells(current))
    init = [Chain.init(l) for l in links]

    def expanding(chain: Chain):
        yield from expand_alc(chain, links)

    def matching(chain: Chain):
        return match_loop(chain) or match_rope(chain)

    def canceling(chain: Chain):
        return len(chain) >= MAX_LENGTH

    for chain in search_breadth(init, expanding, matching, canceling):
        if match_loop(chain):
            res = tuple(resolve_loop(current, chain))
        elif match_rope(chain):
            res = tuple(resolve_rope(current, chain))
        else:
            res = None

        if not res:
            continue  # if didn't work

        yield from res
        break  # on first worked

## Grouping Links

Disjoint groups within a unit form $\veebar$ relation. All other groups within unit can form $\barwedge$


In [ ]:
def search_hard_groups(board: Board) -> Iterable[HLink]:
    for box in Zone.Allbox():
        drafts = set(draftborhood(board, box))
        counts = count_digits(drafts)
        for dig in counts.keys():  # including cnt = 1
            # print(box, dig, counts[dig])
            groups = tuple(scan_zonegroups(board, box, dig))

            # intra-box links
            # for g1, g2 in itercomb(groups, 2):
            #     # print(g1.subzone, g2.subzone, len(g1.cells & g2.cells), len(g1.cells | g2.cells), not (g1.cells & g2.cells) and (g1.cells | g2.cells) == drafts)
            #     if not (g1.cells & g2.cells) and len(g1.cells | g2.cells) == counts[dig]:
            #         yield HLink((g1.node(), g2.node()))

            # side links
            for g1 in groups:
                [side] = set(g1.zones) - {box}
                sidegroups = tuple(scan_zonegroups(board, side, dig))  # this contains g1
                sidegroups = tuple(filter(lambda g: g.cells != g1.cells, sidegroups))
                print(box, dig, g1.zones[0], side, *map(str, g1.cells))
                for g2 in sidegroups:
                    print(*map(str, g2.cells), end=", ")
                print()
                if len(sidegroups) == 2:
                    g1_, g2_ = sidegroups
                    yield HLink((g1_.node(), g2_.node()))

In [ ]:
def check_gsoft(n1: Node, n2: Node):
    assert n1 != n2
    if n1.dig == n2.dig:
        return len(set(visibility(n1.zone, n2.zone))) > 0

## Grouping Chains

All the logic of chains applies to grouping nodes.


## A puzzle


In [ ]:
from utils import bparse, fillempty

puzzle = bparse("""
....1.4..
..9....5.
.67......
8....51..
...6...7.
.........
.8.7..3..
2..9.....
5........
""")

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
puzzle = await solve_silent(
    puzzle,
    open_singles,
    hidden_singles,
)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    open_singles,
    hidden_singles,
    locked_groups,
    open_mults_(2),
    hidden_mults_(2),
    open_mults_(3),
    hidden_mults_(3),
    open_mults_(4),
    hidden_mults_(4),
    open_mults_(5),
    hidden_mults_(5),
    chains,
    filtout={"open_singles", "hidden_singles"},
)

### GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Enum, Bool, Dict
from canvas import SudokuCanvas

In [ ]:
def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


# TODO: make it cancellable somehow

In [ ]:
class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Enum(["INCOMPLETE", "SOLVED", "BROKEN"])
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Node))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.status = validate(self.puzzle)
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for lnk in self.links:
                self._highlight_link(lnk, "blue")

            for node in self.empties:
                self._highlight_node(node, "pink")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for node in self.targets:
                self._highlight_node(node, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in iter(node.zone):
            self._canvas.highlight_segment(loc, node.dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        t1, t2 = lnk
        if t1.zone.is_cellular and t2.zone.is_cellular:
            self._canvas.highlight_link(
                t1.zone.loc(),
                t1.dig,
                t2.zone.loc(),
                t2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            t1locs = tuple(iter(t1.zone))
            l1mid = Loc(
                sum(l.r for l in t1locs) // len(t1locs),
                sum(l.c for l in t1locs) // len(t1locs),
            )
            t2locs = tuple(iter(t2.zone))
            l2mid = Loc(
                sum(l.r for l in t2locs) // len(t2locs),
                sum(l.c for l in t2locs) // len(t2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                t1.dig,
                l2mid,
                t2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig, lmax, node.dig, style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False


# GUI meta-widget


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [ ]:
debug_view = w.Output()
gui = GUI()

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial
    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, resolution, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(resolution)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: "
                if resolution.castaways:
                    resolving += f"-= {len(resolution.castaways)}"
                if resolution.finals:
                    resolving += f"== {len(resolution.finals)}"
                gui.resolving = resolving
                render_resolution(resolution)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    with hold_canvas():
        gui.targets = res.castaways if res.castaways else set()

        gui.anchors = res.highlights.get("anchors", set())
        gui.empties = res.highlights.get("empties", set())

        if "chain" in res.highlights:
            gui.links = set(res.highlights["chain"])
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def clear_resolution():
    gui.reset_highlights()

In [ ]:
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle

In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        open_singles,
        hidden_singles,
        locked_groups,
        open_mults_(2),
        hidden_mults_(2),
        open_mults_(3),
        hidden_mults_(3),
        open_mults_(4),
        hidden_mults_(4),
        open_mults_(5),
        hidden_mults_(5),
        chains,
        # gchains,
        filtout={"open_singles", "hidden_singles"},
    )
)

In [ ]:
task

In [ ]:
task.cancel()